# WI26 업종와 국내 금융지표의 일별 관계 분석

- 원자료 기간: 2023-11-06 ~ 2026-11-05
- 수익률 분석 기간: 2023-11-07 ~ 2026-11-05
- 대상: WI26 26개 업종, 국내 금융지표 6개
- 기본모형: 업종 수익률 ~ 상수 + KOSPI 수익률
- 확장모형: 기본모형 + 금융지표 하나
- 추정: OLS, HAC 시차 5, Bartlett, 소표본 보정
- 검정: 양측 5%, 전체 156개 금융지표 계수에 BY 보정
- 목적: 과거 동시점 관계 설명
- 추가 확인 예정: HAC 시차 1·10에서 결과의 민감도

## 1. 실행 준비

README의 환경 설치 안내를 먼저 확인하세요. 원자료는 포함하지 않았습니다. `data/sample_input.csv`는 형식 설명용 가상 자료이며 분석 실행용이 아닙니다.

## 2. 경로·기간·파라미터 설정

| 파라미터 | 의미 |
|---|---|
| `START`, `END` | 회귀에 사용할 수익률 날짜 범위 |
| `HAC_LAGS=5` | 표준오차 계산 시 5개 관측 시차까지의 자기상관을 고려 |
| `ALPHA=0.05` | 통계적 유의성 판단 기준 |
| `fdr_by` | 여러 검정을 동시에 할 때 사용하는 Benjamini–Yekutieli 보정 |
| `EXPECTED_SECTORS`, `EXPECTED_DAYS` | 다른 자료나 누락된 자료를 실수로 분석하지 않도록 하는 검사 기준 |
| `INPUT_SHA256` | 파일 내용이 조금만 달라도 달라지는 지문 |

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform

import numpy as np
import pandas as pd
import scipy
import statsmodels
import statsmodels.api as sm

from IPython.display import display
from statsmodels.stats.multitest import multipletests

# 1. 입력 자료 경로
# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "requirements_analysis.txt").is_file():
    raise ValueError("market-analysis 또는 그 안의 notebooks 폴더에서 실행하세요.")
INPUT_DIR = PROJECT_ROOT / "data"
INPUT_FILE = INPUT_DIR / "regression_inputs_domestic.csv"
if not INPUT_FILE.is_file():
    raise FileNotFoundError(
        "data/regression_inputs_domestic.csv가 필요합니다. "
        "sample_input.csv는 형식 설명용이며 분석에 사용할 수 없습니다."
    )

# 2. 수익률 분석 기간
# 2023-11-06은 첫 수익률을 계산하는 기준일입니다.
START = "2023-11-07"
END = "2026-11-05"

# 3. 회귀 설정
Y = "sector_return_pct"
MARKET = "kospi_return_pct"

HAC_LAGS = 5
ALPHA = 0.05
MULTIPLE_TEST_METHOD = "fdr_by"

# 현재 고정된 자료가 맞는지 확인할 기준
EXPECTED_SECTORS = 26
EXPECTED_DAYS = 729

# 4. 금융지표 이름과 단위
FACTORS = {
    "KR3Y_change": {
        "name": "국고채 3년",
        "unit": "bp",
    },
    "KR10Y_change": {
        "name": "국고채 10년",
        "unit": "bp",
    },
    "KR_CORP_AA3Y_change": {
        "name": "회사채 AA- 3년",
        "unit": "bp",
    },
    "KR_CREDIT_change": {
        "name": "신용스프레드",
        "unit": "bp",
    },
    "KR_CURVE_change": {
        "name": "장단기 금리차",
        "unit": "bp",
    },
    "USDKRW_change": {
        "name": "달러/원",
        "unit": "%",
    },
}

# 5. 결과 경로 설정 (폴더는 검증·분석 완료 후 생성)
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = PROJECT_ROOT / "results" / RUN_ID

# 6. 입력 파일 지문
# 실행에 사용한 입력 파일의 동일성을 확인합니다.
INPUT_SHA256 = hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest()

pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")

print("입력 파일:", INPUT_FILE)
print("분석 기간:", START, "~", END)
print("HAC 시차:", HAC_LAGS)
print("결과 폴더:", OUTPUT_DIR)
print("입력 SHA256:", INPUT_SHA256)

## 3. CSV 읽기와 데이터 검증


In [ ]:
df = pd.read_csv(
    INPUT_FILE,
    encoding="utf-8-sig",
    dtype={"sector_code": str, "sector_name": str},
    parse_dates=["date", "previous_sector_date"],
)

numeric_cols = [Y, MARKET, *FACTORS.keys()]
required_cols = [
    "date",
    "previous_sector_date",
    "sector_code",
    "sector_name",
    *numeric_cols,
]

missing_cols = sorted(set(required_cols) - set(df.columns))
if missing_cols:
    raise ValueError(f"필수 칼럼이 없습니다: {missing_cols}")

df = df.loc[
    df["date"].between(pd.Timestamp(START), pd.Timestamp(END))
].copy()

df = df.sort_values(["sector_code", "date"]).reset_index(drop=True)

if df.empty:
    raise ValueError("설정한 기간에 자료가 없습니다.")

if df[required_cols].isna().any().any():
    raise ValueError("필수 칼럼에 결측치가 있습니다.")

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="raise")

if not np.isfinite(df[numeric_cols].to_numpy(dtype=float)).all():
    raise ValueError("숫자 칼럼에 무한대 또는 유효하지 않은 값이 있습니다.")

if df.duplicated(["sector_code", "date"]).any():
    raise ValueError("동일 업종·날짜가 중복됩니다.")

if not (df["previous_sector_date"] < df["date"]).all():
    raise ValueError("수익률 시작일과 종료일 순서에 문제가 있습니다.")

if df["sector_code"].nunique() != EXPECTED_SECTORS:
    raise ValueError("예상한 26개 업종와 다릅니다.")

# 같은 날짜에는 모든 업종에 같은 시장·금융지표 값이 있어야 합니다.
shared_cols = ["previous_sector_date", MARKET, *FACTORS.keys()]
if df.groupby("date")[shared_cols].nunique().ne(1).any().any():
    raise ValueError("같은 날짜의 공통 지표 값이 업종마다 다릅니다.")

all_dates = pd.DatetimeIndex(sorted(df["date"].unique()))

if len(all_dates) != EXPECTED_DAYS:
    raise ValueError(f"예상한 729거래일과 다릅니다: {len(all_dates)}")

if (
    all_dates.min() != pd.Timestamp(START)
    or all_dates.max() != pd.Timestamp(END)
):
    raise ValueError("분석 시작일 또는 종료일이 설정과 다릅니다.")

for sector_code, g in df.groupby("sector_code"):
    sector_dates = pd.DatetimeIndex(g["date"])
    if not sector_dates.equals(all_dates):
        raise ValueError(f"{sector_code}: 다른 업종와 날짜가 다릅니다.")
    if g["sector_name"].nunique() != 1:
        raise ValueError(f"{sector_code}: 업종 이름이 일관되지 않습니다.")

audit = (
    df.groupby(["sector_code", "sector_name"], as_index=False)
    .agg(
        n_obs=("date", "size"),
        start_date=("date", "min"),
        end_date=("date", "max"),
    )
)

print(f"검증 완료: {len(df):,}행")
print(f"업종: {df['sector_code'].nunique()}개")
print(f"거래일: {df['date'].nunique()}일")

display(audit)
display(df.head())

## 4. 회귀 실행 함수 정의

여기서 중요한 설정은 다음과 같습니다.
- add_constant: 상수항 $\alpha$를 포함합니다.
- missing="raise": 결측치를 몰래 제거하지 않고 오류를 냅니다.
- use_correction=True: HAC 공분산에 소표본 보정을 적용합니다.
- use_t=False: 계수 검정에 정규분포 근사를 사용합니다. 이후 결과표의 검정통계량은 z입니다.
- RMSE: 실제 수익률과 모형의 적합값 사이 오차 크기입니다. 여기서는 표본 내 적합 오차입니다.
HAC는 계수와 R²를 바꾸는 설정이 아니라 표준오차·검정통계량·p값·신뢰구간을 계산하는 방식입니다.

In [ ]:
def fit_ols_hac(data, x_cols):
    """상수항을 포함한 OLS + HAC 표준오차."""
    X = sm.add_constant(
        data[x_cols].astype(float),
        has_constant="add",
    )
    y = data[Y].astype(float)

    # 변수 사이의 정확한 선형관계 검사
    if np.linalg.matrix_rank(X.to_numpy()) < X.shape[1]:
        raise ValueError(f"설명변수에 정확한 선형관계가 있습니다: {x_cols}")

    if len(data) <= HAC_LAGS + X.shape[1]:
        raise ValueError("표본 수가 너무 적습니다.")

    return sm.OLS(y, X, missing="raise").fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": HAC_LAGS,
            "kernel": "bartlett",
            "use_correction": True,
        },
        use_t=False,
    )


def in_sample_rmse(model):
    """표본 내 잔차의 제곱평균제곱근."""
    return float(np.sqrt(np.mean(np.asarray(model.resid) ** 2)))


print("회귀 함수 준비 완료")

## 5. 26개 업종 × 6개 지표 회귀 실행


In [ ]:
comparison_rows = []
coefficient_rows = []
fitted_models = {}

for sector_code, g in df.groupby("sector_code", sort=True):
    g = g.sort_values("date").copy()
    sector_name = g["sector_name"].iloc[0]

    for factor, info in FACTORS.items():
        # 기본·확장모형 모두 정확히 같은 행 사용
        sample = g[
            ["date", Y, MARKET, factor]
        ].copy()

        base = fit_ols_hac(sample, [MARKET])
        expanded = fit_ols_hac(sample, [MARKET, factor])

        if not (base.nobs == expanded.nobs == len(sample)):
            raise ValueError("기본·확장모형의 관측 수가 다릅니다.")

        key = (sector_code, factor)
        fitted_models[key] = {
            "sample": sample,
            "baseline": base,
            "expanded": expanded,
        }

        ci = expanded.conf_int(alpha=ALPHA)

        comparison_rows.append({
            "sector_code": sector_code,
            "sector_name": sector_name,
            "factor": factor,
            "factor_name": info["name"],
            "factor_unit": info["unit"],
            "start_date": sample["date"].min(),
            "end_date": sample["date"].max(),
            "n_obs": len(sample),

            "corr_raw": sample[Y].corr(sample[factor]),
            "corr_market_factor": sample[MARKET].corr(sample[factor]),

            "beta_factor": expanded.params[factor],
            "se_factor_hac": expanded.bse[factor],
            "z_factor_hac": expanded.tvalues[factor],
            "p_factor_hac": expanded.pvalues[factor],
            "ci_low": ci.loc[factor, 0],
            "ci_high": ci.loc[factor, 1],

            "beta_market_baseline": base.params[MARKET],
            "beta_market_expanded": expanded.params[MARKET],

            "r2_baseline": base.rsquared,
            "r2_expanded": expanded.rsquared,
            "delta_r2": expanded.rsquared - base.rsquared,

            "adj_r2_baseline": base.rsquared_adj,
            "adj_r2_expanded": expanded.rsquared_adj,
            "delta_adj_r2": (
                expanded.rsquared_adj - base.rsquared_adj
            ),

            "rmse_baseline": in_sample_rmse(base),
            "rmse_expanded": in_sample_rmse(expanded),
            "rmse_reduction": (
                in_sample_rmse(base) - in_sample_rmse(expanded)
            ),
        })

        # 기본·확장모형의 모든 계수를 별도 표에 저장
        for model_name, model in [
            ("baseline", base),
            ("expanded", expanded),
        ]:
            model_ci = model.conf_int(alpha=ALPHA)

            for term in model.params.index:
                coefficient_rows.append({
                    "sector_code": sector_code,
                    "sector_name": sector_name,
                    "factor": factor,
                    "model": model_name,
                    "term": term,
                    "coef": model.params[term],
                    "se_hac": model.bse[term],
                    "z_hac": model.tvalues[term],
                    "p_hac": model.pvalues[term],
                    "ci_low": model_ci.loc[term, 0],
                    "ci_high": model_ci.loc[term, 1],
                    "n_obs": int(model.nobs),
                })

results = pd.DataFrame(comparison_rows)
coefficients = pd.DataFrame(coefficient_rows)

if len(results) != EXPECTED_SECTORS * len(FACTORS):
    raise ValueError("예상한 업종·지표 조합 수와 다릅니다.")
if not results["n_obs"].eq(EXPECTED_DAYS).all():
    raise ValueError("조합별 표본 수가 예상과 다릅니다.")

print(f"완료: {len(results)}개 업종·지표 조합")
display(results.head())

## 6. 다중검정 보정과 결과표 확인


156번 검정하면 우연히 작은 p값이 나올 가능성도 커집니다. 전체 156개 금융지표 계수의 p값을 한 번에 보정합니다.

이번에는 업종·지표 사이 의존성을 고려해 비교적 보수적인 BY 방식을 사용합니다. 구현은 statsmodels 다중검정 공식 문서를 따릅니다.

In [ ]:
p_values = results["p_factor_hac"].to_numpy()

if not np.isfinite(p_values).all():
    raise ValueError("금융지표 계수의 p값에 유효하지 않은 값이 있습니다.")

reject, adjusted_p, _, _ = multipletests(
    p_values,
    alpha=ALPHA,
    method=MULTIPLE_TEST_METHOD,
)

results["p_factor_by"] = adjusted_p
results["significant_raw_5pct"] = (
    results["p_factor_hac"] < ALPHA
)
results["significant_by_5pct"] = reject

# 양수: 정방향 관계, 음수: 역방향 관계
results["direction"] = np.select(
    [
        results["beta_factor"] > 0,
        results["beta_factor"] < 0,
    ],
    ["positive", "negative"],
    default="zero",
)

# 표시 순서일 뿐, 최종 지표 선정 점수는 아닙니다.
results = results.sort_values(
    ["p_factor_by", "delta_adj_r2"],
    ascending=[True, False],
).reset_index(drop=True)

summary_cols = [
    "sector_code",
    "sector_name",
    "factor_name",
    "n_obs",
    "corr_raw",
    "beta_factor",
    "ci_low",
    "ci_high",
    "p_factor_hac",
    "p_factor_by",
    "delta_adj_r2",
    "significant_by_5pct",
]

print(
    "보정 전 유의 조합:",
    int(results["significant_raw_5pct"].sum()),
)
print(
    "BY 보정 후 유의 조합:",
    int(results["significant_by_5pct"].sum()),
)

display(results[summary_cols])

# 업종 × 금융지표 형태의 표
beta_table = results.pivot(
    index=["sector_code", "sector_name"],
    columns="factor_name",
    values="beta_factor",
)

adjusted_p_table = results.pivot(
    index=["sector_code", "sector_name"],
    columns="factor_name",
    values="p_factor_by",
)

print("업종별 금융지표 계수")
display(beta_table)

print("업종별 BY 보정 p값")
display(adjusted_p_table)

## 7. 특정 업종·지표의 상세 회귀표 확인

다른 조합을 보려면 두 값만 바꾸면 됩니다.
- 에너지: WI100
- 조선: WI240
- 반도체: WI620
- 국고채 3년: KR3Y_change
- 장단기 금리차: KR_CURVE_change
상세 회귀표의 핵심 항목입니다.

In [ ]:
SELECT_SECTOR = "WI620"          # 반도체
SELECT_FACTOR = "USDKRW_change"  # 달러/원 변화율

selected = fitted_models[(SELECT_SECTOR, SELECT_FACTOR)]

display(
    results.loc[
        (results["sector_code"] == SELECT_SECTOR)
        & (results["factor"] == SELECT_FACTOR),
        summary_cols,
    ]
)

print("기본모형")
print(selected["baseline"].summary())

print("\n확장모형")
print(selected["expanded"].summary())

## 8. 결과·실행 조건 저장


In [ ]:
# 저장 전 입력 파일의 동일성을 마지막으로 확인합니다.
if hashlib.sha256(INPUT_FILE.read_bytes()).hexdigest() != INPUT_SHA256:
    raise ValueError("실행 중 입력 파일이 변경되었습니다.")
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)

results.to_csv(
    OUTPUT_DIR / "regression_comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

coefficients.to_csv(
    OUTPUT_DIR / "regression_coefficients.csv",
    index=False,
    encoding="utf-8-sig",
)

audit.to_csv(
    OUTPUT_DIR / "sample_audit.csv",
    index=False,
    encoding="utf-8-sig",
)

beta_table.to_csv(
    OUTPUT_DIR / "sector_factor_beta.csv",
    encoding="utf-8-sig",
)

adjusted_p_table.to_csv(
    OUTPUT_DIR / "sector_factor_p_by.csv",
    encoding="utf-8-sig",
)

package_versions = {
    "pandas": pd.__version__,
    "numpy": np.__version__,
    "scipy": scipy.__version__,
    "statsmodels": statsmodels.__version__,
}

manifest = {
    "run_id": RUN_ID,
    "input_file": INPUT_FILE.relative_to(PROJECT_ROOT).as_posix(),
    "input_sha256": INPUT_SHA256,
    "analysis_start": START,
    "analysis_end": END,
    "sector_count": int(df["sector_code"].nunique()),
    "observations_per_sector": EXPECTED_DAYS,
    "target": Y,
    "market": MARKET,
    "factors": FACTORS,
    "baseline": "sector_return_pct ~ const + kospi_return_pct",
    "expanded": (
        "sector_return_pct ~ const + kospi_return_pct + one_factor"
    ),
    "estimator": "OLS",
    "covariance": "HAC",
    "hac_maxlags": HAC_LAGS,
    "hac_kernel": "bartlett",
    "hac_use_correction": True,
    "use_t": False,
    "alpha": ALPHA,
    "multiple_testing_method": MULTIPLE_TEST_METHOD,
    "multiple_testing_family": "all 156 added-factor coefficients",
    "missing_policy": "raise; no imputation",
    "outlier_policy": "no deletion or winsorization",
    "analysis_purpose": "retrospective contemporaneous association",
    "publication_times_verified": False,
    "out_of_sample_evaluation": False,
    "python_version": platform.python_version(),
    "packages": package_versions,
}

(OUTPUT_DIR / "run_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

requirements = "\n".join(
    f"{name}=={version}"
    for name, version in package_versions.items()
)

(OUTPUT_DIR / "requirements_analysis.txt").write_text(
    requirements + "\n",
    encoding="utf-8",
)


print("저장 완료:", OUTPUT_DIR)

## 9. 결과 파일 읽기

| 파일 | 용도 |
|---|---|
| regression_comparison.csv | 156개 조합의 계수·유의성·추가 설명력 |
| regression_coefficients.csv | 기본·확장모형의 모든 계수 |
| sample_audit.csv | 업종별 관측 수와 분석 기간 |
| sector_factor_beta.csv | 업종 × 지표 계수 |
| sector_factor_p_by.csv | 업종 × 지표의 BY 보정 p값 |
| run_manifest.json | 입력 지문, 기간, 모형, 파라미터, 환경 |
| requirements_analysis.txt | 실행 시 패키지 버전 |

위 파일은 전체 실행 후 `results/<실행 ID>/`에 생성됩니다. 배포본에 포함된 기존 결과는 `results/` 바로 아래의 두 CSV입니다.

결과를 볼 때는 `beta_factor`, `p_factor_by`, `delta_adj_r2`를 함께 확인하세요. `regression_coefficients.csv`의 p값은 보정 전 값이며 기본모형 계수는 비교 지표별로 반복 저장됩니다.

이 결과는 같은 기간의 관계를 설명하며 미래 예측력이나 인과관계를 검증하지 않습니다. 해석의 한계와 후속 검토는 README를 참고하세요.
